# Fine-Tune Phi-3 Mini 3.8B with LoRA/QLoRA

This notebook fine-tunes **Phi-3 Mini 3.8B** (128k context) on custom data using LoRA, then saves the adapter for quantization.

**Requirements:**
- Google Colab (free T4 GPU works)
- ~8GB VRAM (4-bit quantization)
- Your own training data in JSONL format

**Output:** LoRA adapter weights (~20-50MB) that can be merged and quantized.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 bitsandbytes>=0.43.0 datasets>=2.19.0 accelerate>=0.30.0 huggingface_hub

In [ ]:
# Cell 2: Mount Google Drive (optional, for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/phi3-mini-finetune'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"  # 3.8B params, 128k context
LORA_R = 16                # LoRA rank (higher = more capacity, more VRAM)
LORA_ALPHA = 32            # LoRA alpha (typically 2x rank)
LORA_DROPOUT = 0.05        # LoRA dropout
MAX_SEQ_LENGTH = 2048      # Max sequence length (reduce if OOM)
BATCH_SIZE = 4              # Batch size (reduce if OOM)
GRAD_ACCUM = 4              # Gradient accumulation steps
LEARNING_RATE = 2e-4        # Learning rate
NUM_EPOCHS = 3              # Number of epochs
WARMUP_RATIO = 0.1          # Warmup ratio
SAVE_STEPS = 100            # Save checkpoint every N steps
LOGGING_STEPS = 10          # Log every N steps

print(f"Model: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Batch size: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Cell 4: Load model in 4-bit (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model.num_parameters()/1e9:.2f}B parameters")
print(f"Vocabulary size: {len(tokenizer)}")
print(f"Device map: {model.hf_device_map}")

In [ ]:
# Cell 5: Configure LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 6: Load training data
from datasets import load_dataset

# Option A: Load from local JSONL file
# Upload your JSONL file to Colab first, then:
# dataset = load_dataset('json', data_files='/content/your_training_data.jsonl', split='train')

# Option B: Load from a HuggingFace dataset
# dataset = load_dataset('your_username/your_dataset', split='train')

# Option C: Use sample data for testing
dataset = load_dataset('json', data_files='../data/sample_training_data.jsonl', split='train')

print(f"Dataset size: {len(dataset)} examples")
print(f"Sample: {dataset[0]}")

In [ ]:
# Cell 7: Format data for Phi-3
def format_chatml(example):
    """Format messages into ChatML template for Phi-3."""
    messages = example['messages']
    
    # Build ChatML formatted string
    prompt = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        if role == 'system':
            prompt += f"<|system|>\n{content}<|end|>\n"
        elif role == 'user':
            prompt += f"<|user|>\n{content}<|end|>\n"
        elif role == 'assistant':
            prompt += f"<|assistant|>\n{content}<|end|>\n"
    
    return {"text": prompt}

dataset = dataset.map(format_chatml)
print(f"Formatted sample:\n{dataset[0]['text'][:500]}...")

In [ ]:
# Cell 8: Configure trainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    max_seq_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print("Trainer configured. Ready to train.")

In [ ]:
# Cell 9: Train!
import time

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print(f"\nTraining complete!")
print(f"Time: {elapsed/60:.1f} minutes")

In [ ]:
# Cell 10: Save LoRA adapter
ADAPTER_PATH = f"{OUTPUT_DIR}/phi3-mini-lora-adapter"
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

import os
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f)) 
    for f in os.listdir(ADAPTER_PATH) 
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
)
print(f"Adapter saved to: {ADAPTER_PATH}")
print(f"Adapter size: {adapter_size / 1024 / 1024:.1f} MB")

In [ ]:
# Cell 11: Test inference (optional)
def generate_response(prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Test it
response = generate_response("What is machine learning?")
print(f"Response: {response}")

In [ ]:
# Cell 12: Upload to HuggingFace Hub (optional)
# Uncomment and set your repo name to upload:
#
# from huggingface_hub import login
# login()  # Will prompt for token
#
# HF_REPO = "your_username/phi3-mini-lora-custom"
# model.push_to_hub(HF_REPO, use_auth_token=True)
# tokenizer.push_to_hub(HF_REPO, use_auth_token=True)
# print(f"Uploaded to: https://huggingface.co/{HF_REPO}")

## Next Steps

1. **Download the adapter** from Google Drive: `phi3-mini-finetune/phi3-mini-lora-adapter/`
2. **Quantize** using the scripts in `../scripts/`
3. **Run locally** with Ollama or llama.cpp

See `../README.md` for full instructions.